# IAN 603 · Assignment 10
# Advanced SQL + Views

**Modules:** 11 and 12  
**Points:** 20

This assignment is completed entirely inside this notebook.

## Before you begin

Please review the Module 11 and Module 12 materials before starting the assignment:

- [Review Module 11: Advanced SQL](https://ali2449.github.io/IAN-603-Preparing-Data-for-Analytics-/s_module11.html)
- [Review Module 12: Views & Modular Queries](https://ali2449.github.io/IAN-603-Preparing-Data-for-Analytics-/s_module12.html)

The assignment uses the same ideas from those modules, so reviewing the examples first will make the questions easier to complete.

You will:

1. connect to PostgreSQL,
2. create a small assignment database,
3. write SQL in the provided answer cells,
4. run every answer,
5. keep the query results visible in the notebook,
6. submit the completed `.ipynb` file.

## Important submission rule

Your submitted notebook must contain both your SQL code and the output produced when you run it.

**Do not clear the outputs before submitting.**

You do not need screenshots for this assignment.


## Assignment topics

### Module 11
- Common Table Expressions
- window functions
- ranking
- `LAG`
- running totals

### Module 12
- standard views
- reusable analytics interfaces
- CTEs above views
- materialized views
- snapshot freshness
- `REFRESH MATERIALIZED VIEW`


## How to work through the assignment

Complete the notebook from top to bottom.

For each question:

1. review the related Module 11 or Module 12 example if needed,
2. read the requested output carefully,
3. replace the placeholder SQL,
4. run the answer cell,
5. inspect the result,
6. correct the SQL if needed,
7. leave the final result visible.

At the end, save the notebook before submitting it.


## 0. Install required Python packages


In [ ]:
%pip install -q pandas sqlalchemy psycopg2-binary ipywidgets


## 1. Load the course database helper

You only need this notebook file. The helper is downloaded automatically when needed.

If you want a separate copy:

[Download `ian603_database.py`](https://github.com/ali2449/IAN-603-Preparing-Data-for-Analytics-/raw/refs/heads/MSIA/ian603_database.py)


In [ ]:
from pathlib import Path
import urllib.request

MODULE_FILE = Path("ian603_database.py")
MODULE_URL = (
    "https://raw.githubusercontent.com/"
    "ali2449/IAN-603-Preparing-Data-for-Analytics-/MSIA/"
    "ian603_database.py"
)

if not MODULE_FILE.exists():
    urllib.request.urlretrieve(MODULE_URL, MODULE_FILE)
    print("Course database helper downloaded.")

from ian603_database import IAN603Database

db = IAN603Database()
display_query = db.display_query
execute_query = db.execute_query
execute_script = db.execute_script

print("Database helper ready.")


## 2. Connect to PostgreSQL

Run the next cell, enter your PostgreSQL password, and click **Connect**.


In [ ]:
db.show_connection_panel()


## 3. Check the connection


In [ ]:
if db.connected:
    db.test_connection()
else:
    print("No active PostgreSQL connection.")
    print("Return to the connection panel above and click Connect.")


# Setup · Create the Assignment Tables

The next cell creates three small tables:

- `a10_patients`
- `a10_visits`
- `a10_employees`

It also removes old Assignment 10 views/materialized views so the notebook can be run again safely.

**Run this cell before starting Question 1.**


In [ ]:
setup_sql = "DROP MATERIALIZED VIEW IF EXISTS a10_mv_department_metrics;\nDROP VIEW IF EXISTS a10_vw_patient_visit_summary;\nDROP TABLE IF EXISTS a10_visits CASCADE;\nDROP TABLE IF EXISTS a10_employees CASCADE;\nDROP TABLE IF EXISTS a10_patients CASCADE;\n\nCREATE TABLE a10_patients (\n    patient_id integer PRIMARY KEY,\n    patient_name text NOT NULL,\n    city text NOT NULL,\n    active boolean NOT NULL DEFAULT true\n);\n\nCREATE TABLE a10_visits (\n    visit_id integer PRIMARY KEY,\n    patient_id integer NOT NULL REFERENCES a10_patients(patient_id),\n    department text NOT NULL,\n    admit_time timestamp NOT NULL,\n    discharge_time timestamp,\n    diagnosis text NOT NULL,\n    charge numeric(10,2) NOT NULL CHECK (charge >= 0)\n);\n\nCREATE TABLE a10_employees (\n    employee_id integer PRIMARY KEY,\n    employee_name text NOT NULL,\n    department text NOT NULL,\n    salary numeric(10,2) NOT NULL CHECK (salary > 0)\n);\n\nINSERT INTO a10_patients (patient_id, patient_name, city, active) VALUES\n(101, 'Jordan Lee', 'Greensboro', true),\n(102, 'Morgan Hill', 'Winston-Salem', true),\n(103, 'Casey Reed', 'High Point', true),\n(104, 'Taylor Brooks', 'Greensboro', true),\n(105, 'Avery Scott', 'Burlington', true),\n(106, 'Riley Adams', 'Greensboro', true),\n(107, 'Cameron Price', 'Asheboro', true),\n(108, 'Parker Young', 'High Point', false);\n\nINSERT INTO a10_visits\n(visit_id, patient_id, department, admit_time, discharge_time, diagnosis, charge) VALUES\n(2001, 101, 'Cardiology',       '2026-01-03 08:00', '2026-01-05 11:00', 'Hypertension', 3200.00),\n(2002, 101, 'General Medicine', '2026-03-10 09:00', '2026-03-11 12:00', 'Flu',          1200.00),\n(2003, 102, 'Neurology',        '2026-01-15 14:00', '2026-01-20 10:00', 'Migraine',     5400.00),\n(2004, 102, 'Neurology',        '2026-04-02 11:00', '2026-04-05 09:00', 'Seizure',      6100.00),\n(2005, 103, 'Orthopedics',      '2026-02-11 07:00', '2026-02-14 16:00', 'Fracture',     4800.00),\n(2006, 104, 'Cardiology',       '2026-02-18 10:00', '2026-02-21 13:00', 'Arrhythmia',   7200.00),\n(2007, 104, 'Cardiology',       '2026-05-01 08:30', '2026-05-03 15:00', 'Chest Pain',   3900.00),\n(2008, 105, 'General Medicine', '2026-03-05 12:00', '2026-03-09 09:00', 'Pneumonia',    4600.00),\n(2009, 106, 'Orthopedics',      '2026-03-20 09:00', '2026-03-21 17:00', 'Sprain',        900.00),\n(2010, 106, 'Orthopedics',      '2026-06-15 10:00', '2026-06-18 08:00', 'Joint Pain',   2700.00),\n(2011, 107, 'Cardiology',       '2026-04-12 13:00', NULL,               'Heart Failure', 8300.00),\n(2012, 108, 'General Medicine', '2026-06-01 07:30', '2026-06-02 10:00', 'Dehydration',  1100.00);\n\nINSERT INTO a10_employees (employee_id, employee_name, department, salary) VALUES\n(301, 'Alex Morgan',   'Cardiology',       98000.00),\n(302, 'Jamie Chen',    'Cardiology',       87000.00),\n(303, 'Devon Rivera',  'Cardiology',       87000.00),\n(304, 'Skyler Patel',  'Neurology',       104000.00),\n(305, 'Quinn Foster',  'Neurology',        91000.00),\n(306, 'Reese Bennett', 'Orthopedics',      95000.00),\n(307, 'Drew Coleman',  'Orthopedics',      88000.00),\n(308, 'Hayden Ross',   'General Medicine', 82000.00),\n(309, 'Emerson Ward',  'General Medicine', 78000.00),\n(310, 'Rowan Bell',    'General Medicine', 75000.00);"

execute_script(setup_sql)


## Confirm the assignment data


In [ ]:
display_query("""
SELECT
    (SELECT COUNT(*) FROM a10_patients) AS patients,
    (SELECT COUNT(*) FROM a10_visits) AS visits,
    (SELECT COUNT(*) FROM a10_employees) AS employees;
""", title="Assignment tables created")


## Preview patients


In [ ]:
display_query("""
SELECT *
FROM a10_patients
ORDER BY patient_id;
""", title="a10_patients")


## Preview visits


In [ ]:
display_query("""
SELECT *
FROM a10_visits
ORDER BY admit_time, visit_id;
""", title="a10_visits")


## Preview employees


In [ ]:
display_query("""
SELECT *
FROM a10_employees
ORDER BY department, salary DESC, employee_id;
""", title="a10_employees")


# Question 1 · CTE and Length of Stay
**3 points**

Use a CTE named `completed_visits`.

Inside the CTE:
- keep only visits with a non-NULL `discharge_time`,
- calculate `stay_days` as a numeric number of days.

Then return only visits where `stay_days > 2`.

### Required output columns
- `visit_id`
- `patient_id`
- `department`
- `diagnosis`
- `stay_days`

Round `stay_days` to 2 decimal places and sort longest stay first.

<details><summary>Hint</summary>

A useful calculation is:

`EXTRACT(EPOCH FROM (discharge_time - admit_time)) / 86400.0`

Build the calculation inside the CTE, then filter the CTE in the outer query.

</details>


In [ ]:
q1_sql = """
SELECT
    'Replace this placeholder with your Question 1 query.' AS status;
"""

display_query(q1_sql, title="Question 1 Result")


# Question 2 · Rank Employees Within Department
**3 points**

Use a window function to rank employees by salary **within each department**.

Employees with the same salary should receive the same rank.

### Required output columns
- `employee_id`
- `employee_name`
- `department`
- `salary`
- `salary_rank`

Sort by department, then rank, then employee name.

<details><summary>Hint</summary>

Use:

`RANK() OVER (PARTITION BY ... ORDER BY ...)`

</details>


In [ ]:
q2_sql = """
SELECT
    'Replace this placeholder with your Question 2 query.' AS status;
"""

display_query(q2_sql, title="Question 2 Result")


# Question 3 · Previous Visit with LAG
**3 points**

For each patient, order visits by `admit_time`.

Use `LAG()` to return the previous visit's `discharge_time`.

Then calculate the number of days between the current `admit_time` and the previous discharge.

### Required output columns
- `patient_id`
- `visit_id`
- `admit_time`
- `previous_discharge`
- `days_since_previous`

Round `days_since_previous` to 2 decimals.

The first visit for a patient should naturally have NULL for the previous discharge.

<details><summary>Hint</summary>

First calculate:

`LAG(discharge_time) OVER (PARTITION BY patient_id ORDER BY admit_time)`

A CTE can make the second calculation easier to read.

</details>


In [ ]:
q3_sql = """
SELECT
    'Replace this placeholder with your Question 3 query.' AS status;
"""

display_query(q3_sql, title="Question 3 Result")


# Question 4 · Running Department Charges
**2 points**

Create a running total of visit charges separately for each department.

### Required output columns
- `visit_id`
- `department`
- `admit_time`
- `charge`
- `running_department_charges`

The running total must restart for each department.

Order the window by `admit_time, visit_id`.

<details><summary>Hint</summary>

Use `SUM(charge) OVER (...)`.

For a clear cumulative frame, consider:

`ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`

</details>


In [ ]:
q4_sql = """
SELECT
    'Replace this placeholder with your Question 4 query.' AS status;
"""

display_query(q4_sql, title="Question 4 Result")


# Question 5 · Create a Standard View
**3 points**

Create a view named:

`a10_vw_patient_visit_summary`

The view must return **one row per patient**, including patients with no visits.

### Required view columns
- `patient_id`
- `patient_name`
- `visit_count`
- `total_charges`
- `last_visit_date`

Requirements:
- use a `LEFT JOIN`,
- `visit_count` should be 0 for patients with no visits,
- `total_charges` should be 0 for patients with no visits,
- `last_visit_date` should be the latest admission date.

Use `CREATE OR REPLACE VIEW`.

<details><summary>Hint</summary>

You will need aggregation and `GROUP BY`.

For total charges, `COALESCE(..., 0)` can preserve patients with no visits.

</details>


In [ ]:
q5_sql = """
-- Replace this placeholder with CREATE OR REPLACE VIEW ...
"""

if "CREATE" not in q5_sql.upper():
    print("Complete q5_sql, then run this cell again.")
else:
    execute_script(q5_sql)
    print("View statement executed.")


After creating your view, run the next cell to make sure it returns the expected patient-level summary.

Leave this result visible in your submitted notebook.


In [ ]:
display_query("""
SELECT *
FROM a10_vw_patient_visit_summary
ORDER BY patient_id;
""", title="Check the Patient Visit Summary View")


# Question 6 · Use a View with a CTE
**2 points**

Use **only** `a10_vw_patient_visit_summary` as the data source.

Create a CTE that calculates the average `total_charges` among patients who have at least one visit.

Then return patients whose `total_charges` are above that average.

### Required output columns
- `patient_id`
- `patient_name`
- `visit_count`
- `total_charges`
- `average_patient_charges`

Sort highest `total_charges` first.

<details><summary>Hint</summary>

One CTE can return a single benchmark value:

`AVG(total_charges)`

Then use that benchmark in the outer query.

</details>


In [ ]:
q6_sql = """
SELECT
    'Replace this placeholder with your Question 6 query.' AS status;
"""

display_query(q6_sql, title="Question 6 Result")


# Question 7 · Materialized View Freshness
**4 points**

This question demonstrates that a materialized view stores a **snapshot**.

## Create the materialized view

Create:

`a10_mv_department_metrics`

with **one row per department**.

### Required columns
- `department`
- `visit_count`
- `total_charges`

Use the raw `a10_visits` table as the source.

Before creating it, drop the old materialized view if it exists.

<details><summary>Hint</summary>

Use:

`DROP MATERIALIZED VIEW IF EXISTS ...`

followed by:

`CREATE MATERIALIZED VIEW ... AS SELECT ... GROUP BY ...`

</details>


In [ ]:
q7a_sql = """
-- Replace this placeholder with DROP MATERIALIZED VIEW ...
-- and CREATE MATERIALIZED VIEW ...
"""

if "CREATE MATERIALIZED VIEW" not in q7a_sql.upper():
    print("Complete q7a_sql, then run this cell again.")
else:
    execute_script(q7a_sql)
    print("Materialized view created.")


Run the next cell to check the materialized view you created.

You should see one row per department with the visit count and total charges.


In [ ]:
display_query("""
SELECT *
FROM a10_mv_department_metrics
ORDER BY department;
""", title="Check the Materialized View")


## See what happens when the source data changes

The next cell is provided for you.

It will:
1. remove any previous demo row,
2. refresh your materialized view to establish a clean baseline,
3. insert one new Cardiology visit into the live table,
4. compare the live department totals with the stored materialized-view snapshot.

Run the cell and inspect the result.

The live table should include the new visit, while the materialized view should still show the older snapshot.

> A materialized view does not automatically update when its source table changes.


In [ ]:
mv_exists = execute_query("""
SELECT EXISTS (
    SELECT 1
    FROM pg_matviews
    WHERE schemaname = 'public'
      AND matviewname = 'a10_mv_department_metrics'
) AS exists;
""")

if not bool(mv_exists.iloc[0, 0]):
    print("Complete the materialized view step first.")
else:
    execute_script("""
    DELETE FROM a10_visits
    WHERE visit_id = 2999;

    REFRESH MATERIALIZED VIEW a10_mv_department_metrics;

    INSERT INTO a10_visits
        (visit_id, patient_id, department, admit_time, discharge_time, diagnosis, charge)
    VALUES
        (2999, 108, 'Cardiology',
         '2026-07-10 09:00',
         '2026-07-11 11:00',
         'Follow-up',
         2500.00);
    """)

    display_query("""
    WITH live_metrics AS (
        SELECT
            department,
            COUNT(*) AS visit_count,
            SUM(charge) AS total_charges
        FROM a10_visits
        GROUP BY department
    )
    SELECT
        'LIVE TABLE' AS source,
        department,
        visit_count,
        total_charges
    FROM live_metrics

    UNION ALL

    SELECT
        'MATERIALIZED VIEW' AS source,
        department,
        visit_count,
        total_charges
    FROM a10_mv_department_metrics

    ORDER BY department, source;
    """, title="Live Data vs Stored Snapshot")


## Refresh the materialized view

Now write the PostgreSQL command that refreshes:

`a10_mv_department_metrics`

Run your refresh command, then use the comparison cell below to check the result.

After the refresh, the materialized view should match the live table again.


In [ ]:
q7c_sql = """
-- Write the REFRESH MATERIALIZED VIEW command here.
"""

if "REFRESH MATERIALIZED VIEW" not in q7c_sql.upper():
    print("Complete q7c_sql, then run this cell again.")
else:
    execute_script(q7c_sql)
    print("Materialized view refreshed.")


Run the comparison again.

This time, the live table and the materialized view should show the same department totals. Leave this result visible in your submitted notebook.


In [ ]:
display_query("""
WITH live_metrics AS (
    SELECT
        department,
        COUNT(*) AS visit_count,
        SUM(charge) AS total_charges
    FROM a10_visits
    GROUP BY department
)
SELECT
    'LIVE TABLE' AS source,
    department,
    visit_count,
    total_charges
FROM live_metrics

UNION ALL

SELECT
    'MATERIALIZED VIEW' AS source,
    department,
    visit_count,
    total_charges
FROM a10_mv_department_metrics

ORDER BY department, source;
""", title="Confirm the Refreshed Materialized View")


# Submission Checklist

Before submitting, verify all of the following:

- [ ] I reviewed Modules 11 and 12.
- [ ] I connected to PostgreSQL successfully.
- [ ] I ran the setup cell.
- [ ] I completed Questions 1–7.
- [ ] Every answer cell contains my SQL.
- [ ] Every question has visible output.
- [ ] My standard view was created successfully.
- [ ] My materialized view was created successfully.
- [ ] I demonstrated that the materialized view became stale after the source data changed.
- [ ] I refreshed the materialized view and confirmed that it matched the live data again.
- [ ] I saved the notebook after running the final cells.
- [ ] I did **not** clear the outputs.

## Submit

Submit the completed `.ipynb` file in Canvas.

A good filename is:

`IAN_603_Assignment_10_Completed.ipynb`


# Grading

| Question | Topic | Points |
|---|---|---:|
| 1 | CTE + length of stay | 3 |
| 2 | Ranking window function | 3 |
| 3 | `LAG` | 3 |
| 4 | Running total | 2 |
| 5 | Standard view | 3 |
| 6 | View + CTE | 2 |
| 7 | Materialized view + refresh | 4 |
| **Total** |  | **20** |

Partial credit may be awarded when the SQL approach is reasonable but contains a small syntax or logic error.
